In [ ]:
%cd ./BCG/clinical_data

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

import sys
sys.path.append('/BCG/clinical_data')
from clinical_data_analysis.statistical_analysis.settings import *

import statsmodels.api as sm
import statsmodels.stats.multitest as smm

In [ ]:
DATA_PATH = 'Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'

In [ ]:
!ls Data/

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name='1')

df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]

df.rename(columns={'#':'ID',
                    'תאריך בדיקות דם': 'blood_test_date',
                    'שעת לקיחת דם':'blood_taken_time',
                    'מספר מנות':'num_of_injections',
                    'גיל הפסקת מחזור':'menopause',
                    'אסטרוגן':'estorgen',
                    'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                    'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                    'p-tau_ new_#1':'p-tau_new_#1',
                    'ביקורת אחרי שנה':'checkup after a year',
                    'חיסון':'vaccine',
                    'תרופות':'medications',
                    'הערות':'notes',
                    'Number_of_risk_faktors':'Number_of_risk_factors',
                    'P-tau217':'p-tau217',
                    'צרבונין':'Cerebonin',
                    'Spatial ': 'Spatial'
                    }, inplace=True)

df = df.apply(pd.to_numeric, errors='ignore')
# remove outliers
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)

In [ ]:
df = df[df[MOCA_COL] != '?']
df[MCI] = np.nan
df[MCI] = df[MOCA_COL][df[MOCA_COL].notna()] < MCI_THR
# remove samples with MOCA_score < MIN_MOCA_SCORE
df = df[df[MOCA_COL] >= MIN_MOCA_SCORE]

In [ ]:
df['GFR'].astype(float).dropna()

In [ ]:
df.head()

# Sex and Age distribution

In [ ]:
df = df[((df['Age'] > 64) & (df['Age'] < 80))]

In [ ]:
# Sex pie chart
df_217 = df[df[TAU_217].notna()]
group_counts = df_217['Gender'].value_counts()
total_n = group_counts.sum()
desired_label_order = ['m', 'f']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1)
# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=graph_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
plt.title(f'Sex distribution\n N={df_217[TAU_217].notna().sum()}')
plt.savefig(SAVE_DIR + 'Sex_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Age & Sex distribution — stacked bar chart by gender
age_bins = [65, 70, 75, 80]
age_labels = ['65-70', '70-75', '75-80']
df_217['Age_group'] = pd.cut(df_217['Age'], bins=age_bins, labels=age_labels, right=False)

gender_order = ['m', 'f']
gender_labels = ['Male', 'Female']
age_sex_counts = (
    df_217.groupby(['Age_group', 'Gender'], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(index=age_labels, columns=gender_order, fill_value=0)
)

fig, ax = plt.subplots()
bottom = np.zeros(len(age_labels))
for gender, label, color in zip(gender_order, gender_labels, graph_colors):
    counts = age_sex_counts[gender].values
    ax.bar(age_labels, counts, bottom=bottom, label=label, color=color)
    bottom += counts

ax.set_xlabel('Age group')
ax.set_ylabel('Count')
ax.set_title(f'Age group distribution by sex\n N={df_217[TAU_217].notna().sum()}')
ax.legend()
plt.savefig(SAVE_DIR + 'Age_&_Sex_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')


In [ ]:
# Age distribution
# Create age group bins and assign group labels
age_bins = [65, 70, 75, 80]
age_labels = ['65-70', '70-75', '75-80']
df_217['Age_group'] = pd.cut(df_217['Age'], bins=age_bins, labels=age_labels, right=False)
# Count number of participants in each age group
age_group_counts = df_217['Age_group'].value_counts().reindex(age_labels, fill_value=0)
# Calculate std for each group
age_group_std = df_217.groupby('Age_group')['Age'].std().reindex(age_labels)

# Plot the grouped bar chart with std
fig, ax = plt.subplots()
bars = ax.bar(age_labels, age_group_counts.values, capsize=8, color=healthy_color)#, yerr=age_group_std.values * 2)
ax.set_xlabel('Age group')
ax.set_ylabel('Count')
ax.set_title(f'Age group distribution\n N={df_217[TAU_217].notna().sum()}')
# for rect, count, std in zip(bars, age_group_counts.values, age_group_std.values):
#     ax.text(rect.get_x() + rect.get_width()/2, rect.get_height()+3, f'n={count}\nstd={std:.1f}', 
#             ha='center', va='bottom', fontsize=9)
plt.savefig(SAVE_DIR + 'Age_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')


# Biomarkeres distribution

### p-tau217

In [ ]:
sns.histplot(data=df, x=TAU_217, kde=True, stat='count', element='step')
plt.vlines(TAU_217_POPULATION_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='High p-tau217 threshold', linestyles='dashed')

plt.legend()
plt.title('Histogram of p-tau217 levels')
plt.savefig(SAVE_DIR + 'tau217_hist.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# df[TAU_217_CAT] = pd.cut(df[TAU_217], bins=[0, INT_LOW_TAU_217_THR, INT_HIGH_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate-low', 'Intermediate-high', 'High'])
df[TAU_217_CAT] = pd.cut(df[TAU_217], bins=[0, TAU_217_POPULATION_THR, np.inf], labels=['Low', 'High'])
df['p-tau217 population screen level'] = pd.cut(df[TAU_217], bins=[0, TAU_217_POPULATION_THR, np.inf], labels=['Low', 'High'])
df_217 = df[df[TAU_217_CAT].notna()]
print(df_217.shape)
print(df_217.MCI.sum())

In [ ]:
df_217.ID.to_csv('Data/tau217_ID.csv', index=False)

In [ ]:
group_counts = df_217['p-tau217 population screen level'].value_counts()
total_n = group_counts.sum()
desired_label_order = ['Low', 'High']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1)
# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=graph_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
plt.title(f'p-tau217 population screen level distribution\n N={df_217[TAU_217].notna().sum()}, High threshold = {TAU_217_POPULATION_THR}')
plt.savefig(SAVE_DIR + 'p-tau217_population_screen_level_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
group_counts = df_217[TAU_217_CAT].value_counts()
total_n = group_counts.sum()
# desired_label_order = ['Low', 'Intermediate-low', 'Intermediate-high', 'High']
desired_label_order = ['Low', 'High']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

# explode = (0.1, 0.1, 0.1, 0.1)
explode = (0.1, 0.1)

# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=graph_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
# plt.title(f'p-tau217 categorized distribution\n N={df_217[TAU_217_CAT].notna().sum()}, High threshold = {HIGH_TAU_217_THR}')
plt.title(f'p-tau217 categorized distribution\n N={df_217[TAU_217_CAT].notna().sum()}, High threshold = {HIGH_TAU_217_THR}')
plt.savefig(SAVE_DIR + 'p-tau217_categorized_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
print(df_217.MCI.sum())

In [ ]:
df_217[TAU_217_CAT].value_counts()

### GFAP

In [ ]:
df_gfap = df[df['GFAP'].notna()]
df_gfap[GFAP_CAT] = df_gfap[GFAP].apply(lambda x: 'Low' if x < GFAP_POPULATION_THR else 'High')
df_gfap.shape


In [ ]:
sns.histplot(data=df_gfap, x=GFAP, kde=True, stat='count', element='step')
plt.vlines(GFAP_POPULATION_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='High GFAP threshold', linestyles='dashed')

plt.legend()
plt.title('Histogram of GFAP levels')
plt.savefig(SAVE_DIR + 'GFAP_hist.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
group_counts = df_gfap[GFAP_CAT].value_counts()
total_n = group_counts.sum()
desired_label_order = ['Low', 'High']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1)
# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=graph_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
plt.title(f'GFAP distribution\n N={df_gfap[GFAP].notna().sum()}, High threshold = {GFAP_POPULATION_THR}')
plt.savefig(SAVE_DIR + 'GFAP_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')

### NFL

In [ ]:
def get_nfl_threshold(age):
    if age < 70:
        return HIGH_NFL_65_69_THR
    elif age < 75:
        return HIGH_NFL_70_74_THR
    else:
        return HIGH_NFL_75_80_THR

In [ ]:
df_nfl = df[df['NFL'].notna()]
# df_nfl[NFL_CAT] = df_nfl[NFL].apply(lambda x: 'Low' if x < NFL_POPULATION_THR else 'High')

In [ ]:
df_nfl[NFL_CAT] = 'Low'  # Default to Low
for idx, row in df_nfl.iterrows():
    age = row['Age']
    nfl_value = row[NFL]
    threshold = get_nfl_threshold(age)
    if nfl_value > threshold:
        df_nfl.loc[idx, NFL_CAT] = 'High'

In [ ]:
sns.histplot(data=df_nfl, x=NFL, kde=True, stat='count', element='step')
# plt.vlines(NFL_POPULATION_THR, 0, plt.gca().get_ylim()[1], )

plt.legend()
plt.title('Histogram of NFL levels')
plt.savefig(SAVE_DIR + 'NFL_hist.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
group_counts = df_nfl[NFL_CAT].value_counts()
total_n = group_counts.sum()
desired_label_order = ['Low', 'High']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1)
# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=graph_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
plt.title(f'NFL distribution\n N={df_nfl[NFL].notna().sum()}')
plt.savefig(SAVE_DIR + 'NFL_distribution.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
total_n = group_counts.sum()
desired_label_order = ['Low',  'Intermediate-low', 'Intermediate-high', 'High']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1, 0.1, 0.1)

# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        # autopct=autopct_format_n, # Use the custom function here
        autopct="%1.1f%%", # Use the custom function here
        colors=pie_colors,
        shadow=True,
        explode=explode)



# MOCA and MCI distribution

In [ ]:
explode = (0.1, 0.1)

plt.pie(df[MCI].value_counts(), labels=['Cognitive\nHealthy', 'Cognitive\nImpaired'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
plt.title(f'All subjects: Cognitive healthy vs Impaired distribution\n N={df[MOCA_COL].notna().sum()}')
plt.savefig(SAVE_DIR + 'Cognitive_healthy_vs_Impaired_pie.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# same but only for tau217
explode = (0.1, 0.1)
df_217 = df[df[TAU_217].notna()]
plt.pie(df_217[MCI].value_counts(), labels=['Cognitive\nHealthy', 'Cognitive\nImpaired'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
plt.title(f'p-tau217: Cognitive healthy vs Impaired distribution\n N={df_217[MOCA_COL].notna().sum()}')
plt.savefig(SAVE_DIR + 'Cognitive_healthy_vs_Impaired_pie.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
print(df_217.MCI.value_counts())

In [ ]:
df_217[MCI].value_counts()

In [ ]:
print(df.shape)
sns.histplot(data=df.dropna(subset=MOCA_COL), x=MOCA_COL, kde=True, stat='count', element='step', bins=10)
print(df.shape)
plt.vlines(MCI_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='MCI threshold', linestyles='dashed')

plt.legend(loc='upper left')
plt.title('Histogram of MoCA levels')
plt.savefig(SAVE_DIR + 'MOCA_hist.pdf', bbox_inches='tight', dpi=300, format='pdf')


# Startification by MOCA and p-tau217

In [ ]:
df_217.shape
print(df_217.MCI.value_counts())

In [ ]:
df_217['group'] = 'Healthy'
condition_ad_mci = (df_217[TAU_217_CAT] == 'High') & (df_217[MCI] == True)
condition_pre_clinical_ad = (df_217[TAU_217_CAT] == 'High') & (df_217[MCI] == False)
condition_non_ad_mci = (df_217[TAU_217_CAT] == 'Low') & (df_217[MCI] == True)


df_217.loc[condition_ad_mci, 'group'] = 'AD MCI'
df_217.loc[condition_pre_clinical_ad, 'group'] = 'pre-clinical AD'
df_217.loc[condition_non_ad_mci, 'group'] = 'Non-AD MCI'

In [ ]:
for g in df_217['group'].unique():
    print(g)
    print(df_217[df_217['group'] == g].MCI.sum())

In [ ]:
print(df_217.MCI.sum())
df_217['group'].value_counts()

In [ ]:
group_counts = df_217['group'].value_counts()
total_n = group_counts.sum()
desired_label_order = ['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD']
ordered_group_counts = group_counts.reindex(desired_label_order, fill_value=0)
pie_sizes = ordered_group_counts.values
pie_labels = ordered_group_counts.index.tolist()

def autopct_format_n(pct):
    # Calculate the absolute number from the percentage (pct) and the total (total_n)
    absolute_n = int(round(pct / 100.0 * total_n))
    return f"{absolute_n}"

explode = (0.1, 0.1, 0.1, 0.1)

# plt.figure(figsize=(8, 8))
plt.pie(pie_sizes,
        labels=pie_labels,
        autopct=autopct_format_n, # Use the custom function here
        colors=pie_colors,
        shadow=True,
        explode=explode)

# plt.pie(df_217[TAU_217_CAT].value_counts(), labels=['Healthy', 'Non-AD MCI', 'AD MCI', 'pre-clinical AD'], autopct='%1.1f%%', colors=pie_colors, shadow=True, explode=explode)
plt.title(f'Cohort stratified by p-tau217 and cognitive decline distribution\n N={df_217[TAU_217_CAT].notna().sum()}')
plt.savefig(SAVE_DIR + 'p-tau217_cog_healthy_vs_impaired_pie(4 groups graphs).pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
df_217['group'].value_counts()

# Testing if the prevalence of high p-tau217 increases with age

In [ ]:
# Testing if the prevalence of high p-tau217 increases with age
# bin samples by age groups 5 years apart
bar_color = "#1f77b4"  # Use the blue from previous graphs (matplotlib default blue)
df_217['Age_group'] = pd.cut(df_217['Age'], bins=range(64, 80, 5), labels=['65-70', '70-75', '75-80'])
print(df_217['Age_group'].value_counts())
df_217['numerical_tau_category'] = df_217[TAU_217_CAT].replace({'Low': 0, 'High': 1})
df_217['high_p_tau217'] = (df_217[TAU_217_CAT] == 'High').astype(int)
# plot the prevalence of high p-tau217 in each age group
sns.barplot(data=df_217, x='Age_group', y='high_p_tau217', color=graph_colors[0])
plt.title('Prevalence of high p-tau217 in each age group')
plt.xlabel('Age group')
plt.ylabel('Prevalence of high p-tau217')
plt.savefig(SAVE_DIR + 'p-tau217_prevalence_by_age_group.pdf', bbox_inches='tight', dpi=300, format='pdf')
plt.show()


In [ ]:
# Calculate the fold fold increase between each bin
df_217[df_217[TAU_217_CAT] == 'High']['Age_group'].value_counts()

### Cognitive and behavioral test correlation

In [ ]:
# same but only for high p-tau217 individuals
df_217_high = df_217[df_217[TAU_217] > HIGH_TAU_217_THR]
# Ensure the relevant columns exist and drop rows with missing values
cognitive_behavioral_df = df[[TMT_B_COL, MBI_C_COL, TAU_217]].dropna()
# remove the sample with extreamly high TMT_B
# scatter_df = cognitive_behavioral_df[cognitive_behavioral_df[TMT_B_COL] != cognitive_behavioral_df[TMT_B_COL].max()]
# calculate the correlation between MOCA_score and MBI-C
correlation = scatter_df[TMT_B_COL].corr(scatter_df[MBI_C_COL])
print(f"Correlation between MOCA_score and MBI-C: {correlation}")


In [ ]:
# Create a Normalize object that saturates at 1.5
norm = colors.Normalize(vmin=0, vmax=1, clip=True)

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    scatter_df[TMT_B_COL],
    scatter_df[MBI_C_COL],
    c=scatter_df[TAU_217],
    cmap='Reds',
    alpha=0.8,
    # edgecolor='k',
    norm=norm
)
plt.colorbar(sc, label='p-tau217')
plt.xlabel('TMT_B_COL')
plt.ylabel('MBI-C')
plt.title(f'Scatter plot of TMT_B_COL score vs MBI-C colored by p-tau217\n N={scatter_df.shape[0]} high p-tau217, Correlation: {correlation:.2f}')
plt.grid(True)
plt.show()


In [ ]:
# Histogram of MBI-c scores but only on high p-tau217 individuals with MCI == 0
df_217_high = df_217[df_217[TAU_217] > HIGH_TAU_217_THR]
df_217_high_no_mci = df_217_high[df_217_high[MCI] == 0]
df_217_high_mci = df_217_high[df_217_high[MCI] == 1]

sns.histplot(data=df_217_high_no_mci, x=MBI_C_COL, kde=True, stat='count', element='step')
sns.histplot(data=df_217_high_mci, x=MBI_C_COL, kde=True, stat='count', element='step')
plt.title(f'MBI-C scores distribution in high p-tau217 individuals MCI (N={df_217_high_mci.shape[0]}) vs cognitive healthy (N={df_217_high_no_mci.shape[0]})')
plt.legend()
plt.show()


In [ ]:
# Trait association analysis between MOCA and MBI-C
linear_regerssion_data = df[[MOCA_COL, MBI_C_COL, TAU_217, 'Age', 'Gender']].notna()
model = sm.OLS(linear_regerssion_data[MOCA_COL], linear_regerssion_data[MBI_C_COL]).fit()
model.summary()

# # Add covariates to the model age and sex
# model = sm.OLS(df_217[MOCA_COL], df_217[MBI_C_COL] + df_217['Age'] + df_217['Gender']).fit()
# model.summary()


# Are treated risk factor controled?

After we saw we didn't find any risk factors related to the tau level, we want to check if there are subjects where they reported the risk factors, but it's not controlled. And if we have enough subjects we want to use only them for the risk factor analysis

In [ ]:

def blood_tests_preprocess(df):
    df['HA1C_CAT'] = pd.cut(df['Hemoglobin_A1C'], bins=[0, HA1C_INT, HA1C_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['CHOL_CAT'] = pd.cut(df['Cholesterol'],  bins=[0, CHOL_INT, CHOL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['Triglycerides'].replace({'38..8': 38.8}, inplace=True)
    df['TRG_CAT'] = pd.cut(df['Triglycerides'], bins=[0, TRG_INT, TRG_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['LDL_CAT'] = pd.cut(df['LDL'], bins=[0, LDL_INT, LDL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['CRP'].replace({'<0.4': 0, '<0.5': 0, '<0.6':0}, inplace=True)
    df['SBP'] = df['BP'].str.split('/').str[0].astype(float)
    df['DBP'] = df['BP'].str.split('/').str[1].astype(float)
    # df = df[df['CRP'] != '<1']
    df['CRP'].replace({'<1': np.nan}, inplace=True)
    df['CRP_CAT'] = pd.cut(df['CRP'], bins=[0, CRP_INT, CRP_HIGH, CRP_VERY_HIGH, np.inf], labels=['Low', 'Intermediate', 'High', 'Very High'])
    df['WBC_CAT'] = pd.cut(df['WBC'], bins=[0, WBC_LOW, WBC_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['LYMPH_CAT'] = pd.cut(df['Lymphocytes'], bins=[0, LYMPH_LOW, LYMPH_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])

    return df

def genetic_risk_factors(df):
    df['APOE_genotype'] = df['APOE-SNP'].replace({'APOE3/3':0, 'APO3/3':0, 'APO3/4':1, 'APOE3/4':1, 'APOE4/4':2})
    df['APOE_risk'] = df['APOE-SNP'].replace({'APOE3/3':0, 'APO3/3':0, 'APO3/4':1, 'APOE3/4':1, 'APOE4/4':1})
    df['Gender'].replace({'f':1, 'm':0}, inplace=True)
    return df

df = blood_tests_preprocess(df)
df = genetic_risk_factors(df)

In [ ]:
# Hyperlipidemia
print("Hyperlipidemia")
print((df['Hyperlipidemia'] == 'YES').sum())
print(((df['LDL'] >= 100) & (df['HDL'] <= 45) & (df['Triglycerides'] >= 150)).sum())
print(df[(df['Hyperlipidemia'] == 'YES') & ((df['LDL'] >= 100) & (df['HDL'] <= 45) & (df['Triglycerides'] >= 150))].shape[0] / df[(df['Hyperlipidemia'] == 'YES')].shape[0])

# Hypertension
print("Hypertension")
print((df['Hypertension'] == 'YES').sum())
print(df[(df['SBP'] >= 140) & (df['DBP'] >= 90)].shape[0])
# print(df[(df['SBP'] >= 140) & (df['DBP'] >= 90)].shape[0] / df[(df['Hypertension'] == 'YES')].shape[0])
print(df[(df['Hypertension'] == 'YES') & ((df['SBP'] >= 140) & (df['DBP'] >= 90))].shape[0] / df[(df['Hypertension'] == 'YES')].shape[0])

# Diabetes
print("Diabetes")
print((df['Diabetes'] == 'YES').sum())
print(df[(df['Hemoglobin_A1C'] >= 6.7)].shape[0])
# print(df[(df['Hemoglobin_A1C'] >= 6.1)].shape[0] / df[(df['Diabetes'] == 'YES')].shape[0])
print(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)].shape[0] / df[(df['Diabetes'] == 'YES')].shape[0])
print(f'Out of the uncontrolled diabetes, {(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7) & (df['APOE_risk'] == 1)].shape[0] / df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)].shape[0]) * 100:.2f}% are APOE4 carriers')
print(f'Out of the controlled diabetes, {(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7) & (df['APOE_risk'] == 1)].shape[0] / df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7)].shape[0]) * 100:.2f}% are APOE4 carriers')
print(f'Out of the controlled diabetes, {(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7) & (df['Herpes_zoster_vaccine'] == 'YES')].shape[0] / df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7)].shape[0]) * 100:.2f}% had the herpes zoster vaccine')
print(f'Out of the uncontrolled diabetes, {(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7) & (df['Herpes_zoster_vaccine'] == 'YES')].shape[0] / df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)].shape[0]) * 100:.2f}% had the herpes zoster vaccine')


In [ ]:
diabetes_not_controled = df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)]
diabetes_controled = df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7)]
all_diabetes = df[df['Diabetes'] == 'YES']
only_healthy = df[df['Diabetes'] == 'NO']
alpha = 0.2

sns.histplot(data=only_healthy, x=TAU_217, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=TAU_217, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=TAU_217, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=TAU_217, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('Tau 217')
plt.title('Diabetes Controled vs Not Controled')
plt.savefig(SAVE_DIR + 'Diabetes_Controled_vs_Not_Controled_TAU_217.pdf', bbox_inches='tight', dpi=300, format='pdf')
plt.show()

plt.legend()
plt.title('Diabetes Controled vs Not Controled')
plt.show()

# GFAP
sns.histplot(data=only_healthy, x=GFAP, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=GFAP, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=GFAP, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=GFAP, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('GFAP')
plt.title('Diabetes Controled vs Not Controled')
plt.savefig(SAVE_DIR + 'Diabetes_Controled_vs_Not_Controled_GFAP.pdf', bbox_inches='tight', dpi=300, format='pdf')
plt.show()
#NFL
sns.histplot(data=only_healthy, x=NFL, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=NFL, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=NFL, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=NFL, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('NFL')
plt.title('Diabetes Controled vs Not Controled')
plt.savefig(SAVE_DIR + 'Diabetes_Controled_vs_Not_Controled_NFL.pdf', bbox_inches='tight', dpi=300, format='pdf')
plt.show()

In [ ]:
df.columns